# Analytic prediction vs measured Renyi-2 correlator

Two panels, both built **only from already-computed data** in `experiments/results/`
and `experiments/results/_cache/` (no TEBD or ED is run in this notebook):

1. **Left** — `R(0,100)` vs `epsilon` in the thermodynamic limit (iTEBD), reworked from
   `imps_correlator_vs_epsilon.png` (left subplot only). The per-sample free power-law
   fit is replaced by the **analytic prediction** `R = q/4`, `q = |<11|L''|00>|^2`,
   with **no free parameters** — it uses only `q`, not the measured `R`.
2. **Right** — finite chains vs the infinite-system value, reworked from
   `imps_finite_vs_infinite.png`. The dotted line (previously `R_iTEBD / 2` with an
   *empirical, unexplained* factor of 2) is replaced by the **analytic finite-N
   prediction `R = q/8`**. `CLAUDE.md` now records both `R_iTEBD = q/4` and
   `R_finite = q/8` as derived results, so the factor of 2 between them is exact,
   not fitted.

`q` is reconstructed via `renyi2_swssb.draw_samples()`, which regenerates the L''
operators from their fixed seeds — this is the same deterministic call the existing
pipeline scripts (e.g. `imps_eps_init_grids.report_law`) already use to get `q`; it
performs no simulation and reproduces the exact matrices used to produce the cached
runs (verified below to machine precision).

In [1]:
import os
import pickle
import sys

import numpy as np

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if os.path.basename(ROOT) != "Lindbladian_steady_states":
    ROOT = r"C:\Users\nzier\Lindbladian_steady_states"
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "experiments"))

import renyi2_swssb as ex
import imps_eps_init_grids as grids
import imps_summary_figures as figs

RESULTS_PATH = os.path.join(ex.RESULTS_DIR, "imps_eps_init_grids.pkl")
assert os.path.exists(RESULTS_PATH), "expected cached results, found none"
with open(RESULTS_PATH, "rb") as f:
    results = pickle.load(f)["results"]

SAMPLES = figs.SAMPLES
FINITE_N = figs.FINITE_N
CHI = figs.CHI
R_INF = figs.R_INF
EPSILONS = figs.EPSILONS
print(f"loaded {len(results)} cached iTEBD jobs from {RESULTS_PATH}")

loaded 47 cached iTEBD jobs from C:\Users\nzier\Lindbladian_steady_states\experiments\results\imps_eps_init_grids.pkl


## Reconstruct `q = |<11|L''|00>|^2` at each measured epsilon

Deterministic given the fixed seeds in `renyi2_swssb.draw_samples()` -- no TEBD, no
ED, just re-evaluating the model definition already used to build every cached run.
Sanity-checked against a cached run's stored `L_pp` to confirm it reproduces the
exact operator (not merely a statistically similar one).

In [2]:
def q_of(eps: float) -> list[float]:
    old = ex.EPSILON
    try:
        ex.EPSILON = eps
        samples = ex.draw_samples()
    finally:
        ex.EPSILON = old
    return [abs(s["L_pp"][3, 0]) ** 2 for s in samples]


q_at = {eps: q_of(eps) for eps in EPSILONS}

# Sanity check against a cached finite-N run's stored L'' (bit-for-bit, not a fit).
_check = pickle.load(open(os.path.join(
    ex.CACHE_DIR, "chi128_sample3_zero_N12_chi128.pkl"), "rb"))
_q_cached = abs(_check["L_pp"][3, 0]) ** 2
assert abs(_q_cached - q_at[0.20][3]) < 1e-12, "q reconstruction does not match cache"
print(f"q reconstruction verified against cache to {abs(_q_cached - q_at[0.20][3]):.1e}")
for eps in EPSILONS:
    print(f"  eps={eps:<5}  q = {['%.6e' % v for v in q_at[eps]]}")

q reconstruction verified against cache to 0.0e+00
  eps=0.2    q = ['5.680394e-03', '1.039473e-02', '5.619371e-03', '7.922943e-03', '5.222383e-03', '1.421632e-02', '2.214279e-02', '3.987903e-03', '1.773849e-03', '1.393066e-02']
  eps=0.15   q = ['3.195222e-03', '5.847037e-03', '3.160896e-03', '4.456656e-03', '2.937591e-03', '7.996681e-03', '1.245532e-02', '2.243196e-03', '9.977899e-04', '7.835999e-03']
  eps=0.1    q = ['1.420099e-03', '2.598683e-03', '1.404843e-03', '1.980736e-03', '1.305596e-03', '3.554081e-03', '5.535698e-03', '9.969758e-04', '4.434622e-04', '3.482666e-03']
  eps=0.05   q = ['3.550246e-04', '6.496707e-04', '3.512107e-04', '4.951840e-04', '3.263990e-04', '8.885201e-04', '1.383924e-03', '2.492439e-04', '1.108655e-04', '8.706665e-04']


## Measured values (cached iTEBD trajectories and finite-N runs only)

In [3]:
inf = figs.infinite_values(results)  # R(r=100) per (epsilon, sample), cached trajectories only

devs = []
for s in SAMPLES:
    for eps in EPSILONS:
        y = inf.get((eps, s))
        if y is None:
            continue
        pred = q_at[eps][s] / 4.0
        devs.append(abs(y / pred - 1.0))
devs = np.array(devs)
print(f"R = q/4 prediction vs {len(devs)} measured (epsilon, sample) points:")
print(f"  max  |measured/predicted - 1| = {devs.max():.4%}")
print(f"  mean |measured/predicted - 1| = {devs.mean():.4%}")

R = q/4 prediction vs 40 measured (epsilon, sample) points:
  max  |measured/predicted - 1| = 0.3130%
  mean |measured/predicted - 1| = 0.0779%


## Build the combined figure

Left axis: reworked `imps_correlator_vs_epsilon.png` (left subplot) -- markers are
measured `R`, solid lines are the analytic prediction `R = q/4` (no fit).

Right axes (main + broken-off infinite column): reworked
`imps_finite_vs_infinite.png` -- unchanged except the dotted reference line / star
height, which is now the analytic finite-N law `q/8` rather than the empirical
`R_iTEBD / 2`.

In [4]:
plt = ex._mpl()
cmap = plt.get_cmap("tab10")

fig = plt.figure(figsize=(20, 7.5))
outer = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.25], wspace=0.22)
ax_eps = fig.add_subplot(outer[0])
inner = outer[1].subgridspec(1, 2, width_ratios=[4, 1], wspace=0.04)
ax_main = fig.add_subplot(inner[0])
ax_inf = fig.add_subplot(inner[1])

# ------------------------------------------------------------- left: R vs epsilon
eps_fine = np.linspace(0.045, 0.21, 200)
for s in SAMPLES:
    y = np.array([inf.get((e, s), np.nan) for e in EPSILONS])
    if not np.isfinite(y).all():
        continue
    c = cmap(s % 10)
    pred_fine = (q_at[0.20][s] / 4.0) * (eps_fine / 0.20) ** 2
    ax_eps.loglog(EPSILONS, y, "o", color=c, ms=7, label=f"sample {s}")
    ax_eps.loglog(eps_fine, pred_fine, "-", color=c, lw=1.3, alpha=0.85)

ax_eps.set_xlabel(r"$\epsilon = \|L''\|$")
ax_eps.set_ylabel(rf"$R(0,{R_INF})$   (infinite system)")
ax_eps.set_title(r"$R$ vs $\epsilon$: analytic prediction $R = q/4$, $q=|\langle 11|L''|00\rangle|^2$"
                  "\n" rf"no free parameters — max deviation {devs.max():.3%}", fontsize=11)
ax_eps.grid(True, alpha=0.3, which="both")
ax_eps.legend(fontsize=7, ncol=2)

# ------------------------------------------------------- right: finite vs infinite
plotted: list[float] = []
for s in SAMPLES:
    ys = [figs.finite_value(s, N) for N in FINITE_N]
    xs = [N for N, y in zip(FINITE_N, ys) if y is not None]
    ys = [y for y in ys if y is not None]
    plotted.extend(ys)
    c = cmap(s % 10)
    ax_main.plot(xs, ys, "o-", color=c, ms=6, lw=1.4, label=f"sample {s}")

    pred = q_at[0.20][s] / 8.0  # analytic finite-N law
    plotted.append(pred)
    ax_main.axhline(pred, color=c, ls=":", lw=1.0, alpha=0.55)
    ax_inf.plot([0], [pred], "*", color=c, ms=17, mec="black", mew=0.7)

# --- L''=0 controls (same cached-only sources as imps_finite_vs_infinite.png) ---
base_fin = {N: figs.finite_baseline(N) for N in FINITE_N}
inf_ctrl = figs.infinite_baseline()  # loads the cached x3 control run, no new TEBD
inf_runs = sorted(r["profile"][R_INF] for r in inf_ctrl.get("all_runs", [inf_ctrl]))
floor_top = max(v for v in base_fin.values() if v)
y_lo = 2e-7

ax_main.axhspan(1e-300, floor_top, color="0.55", alpha=0.20, zorder=0)
ax_main.text(0.985, 0.045, r"$L''=0$ finite control (numerical floor)",
             transform=ax_main.transAxes, fontsize=9, style="italic", color="0.25",
             va="bottom", ha="right")
shown = [(N, v) for N, v in base_fin.items() if v and v > y_lo]
ax_main.plot([N for N, _ in shown], [v for _, v in shown], "x--", color="black",
             ms=9, mew=2.0, lw=1.4, zorder=6, label=r"$L''=0$ (finite)")
below = sorted(N for N, v in base_fin.items() if v and v <= y_lo)
if below:
    for N in below:
        ax_main.annotate("", xy=(N, y_lo * 1.25), xytext=(N, y_lo * 6.0),
                          arrowprops=dict(arrowstyle="-|>", color="black", lw=1.6))
    ax_main.text(0.015, 0.045,
                 rf"$L''=0$ at $N={{{', '.join(map(str, below))}}}$ lies below the axis "
                 rf"($\leq 10^{{{int(np.floor(np.log10(max(base_fin[N] for N in below))))}}}$, "
                 r"down to $10^{-112}$ at $N=4$)",
                 transform=ax_main.transAxes, fontsize=8, style="italic", color="0.25")

ax_inf.plot([0.34, 0.34], [inf_runs[0], inf_runs[-1]], "-", color="black",
            lw=2.4, zorder=6, solid_capstyle="butt")
ax_inf.plot([0.34] * len(inf_runs), inf_runs, "_", color="black", ms=14, mew=2.2, zorder=7)
ax_inf.annotate(r"$L''\!=\!0$" "\n" "gapless", xy=(0.34, inf_runs[0]),
                 xytext=(0, -13), textcoords="offset points", fontsize=7.5,
                 ha="center", va="top", color="black", style="italic")

ax_main.set_yscale("log")
ax_main.set_xticks(FINITE_N)
ax_main.set_xlabel("system size $N$ (finite chain, open boundaries)")
ax_main.set_ylabel(r"$R(N/4,\, 3N/4)$")
ax_main.set_title(r"Finite chains: $R$ at quarter/three-quarter sites", fontsize=11)
ax_main.grid(True, alpha=0.3, which="both")
ax_main.legend(fontsize=8, ncol=6, loc="upper center", bbox_to_anchor=(0.5, -0.11),
               frameon=False)

ax_inf.set_yscale("log")
ax_inf.set_xticks([0]); ax_inf.set_xticklabels([r"$N=\infty$"], fontsize=12)
ax_inf.set_xlim(-0.6, 0.6)
ax_inf.set_yticklabels([])
ax_inf.yaxis.set_minor_formatter(plt.matplotlib.ticker.NullFormatter())
ax_inf.yaxis.set_major_formatter(plt.matplotlib.ticker.NullFormatter())
ax_inf.tick_params(axis="y", which="both", labelleft=False, labelright=False)
ax_inf.grid(True, alpha=0.3, which="both")
ax_inf.set_title("iTEBD" "\n" r"(thermodynamic limit)", fontsize=10, pad=14)
ax_inf.set_facecolor("#f2f2f2")

y_hi = 10 ** (np.log10(max(plotted)) + 0.35)
for a in (ax_main, ax_inf):
    a.set_ylim(y_lo, y_hi)

for x, a in ((1.0, ax_main), (0.0, ax_inf)):
    a.spines["right" if a is ax_main else "left"].set_visible(False)
    a.plot([x, x], [0, 1], transform=a.transAxes, color="white", lw=3,
           clip_on=False, zorder=5)

fig.suptitle(r"Renyi-2 SWSSB correlator vs the analytic prediction "
             r"($\epsilon=0.2$, $|neel\rangle$, $\chi=128$ unless noted)", fontsize=13)
fig.text(0.5, 0.015,
         r"Left: solid curves are $R=q/4$, not fitted (the free power-law fit gave $p=1.9991\pm0.0008$)." "\n"
         r"Right: dotted lines / stars are the analytic finite-law $q/8$ — previously $R_{\rm iTEBD}/2$ "
         r"with an empirical, unexplained factor of 2; now $R_{\rm iTEBD}=q/4$ and $R_{\rm finite}=q/8$ are both "
         r"derived, so the factor is exactly 2." "\n"
         rf"$L''\!=\!0$ finite control ($\times$) is a true noise floor. The $L''\!=\!0$ infinite control is not: "
         rf"unperturbed, the model is critical ($A\!+\!A\!\to\!\emptyset$, gapless, $z\!=\!2$).",
         ha="center", fontsize=7.8, style="italic", linespacing=1.5)

fig.subplots_adjust(left=0.055, right=0.985, top=0.87, bottom=0.24, wspace=0.22)

out_path = os.path.join(ex.RESULTS_DIR, "imps_analytic_prediction_summary.png")
#fig.savefig(out_path, dpi=150)
#print(f"saved -> {out_path}")
plt.show()

C:\Users\nzier\AppData\Local\Temp\ipykernel_8496\3578741019.py:121: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
